# K-fold training on Colab

**Runtime → Change runtime type → T4 GPU**

Two options:
- **Cells 1–4** — train directly in this notebook (recommended)
- **Cells 5+** — SSH tunnel from Mac (long sessions, same as local workflow)

In [ ]:
# Cell 1 — clone, install, mount Drive
REPO = "https://github.com/Nontakorn-dev/DernDee_Research.git"
DATA_ROOT = "/content/drive/MyDrive/Dataset/processed/Xy"  # <-- adjust

!git clone -q {REPO} || (cd DernDee_Research && git pull -q)
%cd DernDee_Research
!pip install -q -r shared/requirements.txt

from google.colab import drive
drive.mount("/content/drive")

!python shared/scripts/check_xy_labels.py --channels bilateral --xy-root "{DATA_ROOT}"

In [ ]:
# Cell 2 — run next pending k-fold job (1 job per Colab session)
DATA_ROOT = "/content/drive/MyDrive/Dataset/processed/Xy"

!bash experiments/scripts/run_kfold_colab.sh \
  --data-root "{DATA_ROOT}" \
  --device cuda \
  --lazy \
  --max-jobs 1 \
  --next

In [ ]:
# Cell 3 — batch: one model baseline (15 jobs = 5 folds × 3 seeds)
DATA_ROOT = "/content/drive/MyDrive/Dataset/processed/Xy"
MODEL = "tcn"  # cnn1d | tcn | tinytcn | lstm_gru | transformer

!bash experiments/scripts/run_kfold_colab.sh \
  --data-root "{DATA_ROOT}" \
  --device cuda \
  --lazy \
  --model {MODEL} \
  --phase baseline \
  --max-jobs 15

In [ ]:
# Cell 4 — zip & download runs back to Mac
!zip -qr kfold_runs.zip experiments/*/runs/fold* experiments/compression/runs/fold* 2>/dev/null || true
from google.colab import files
files.download("kfold_runs.zip")

---
## Option B — SSH from Mac (Colab Pro / very long jobs)

Run cells below, copy the `ssh ...` command, then train from Mac Terminal inside Colab VM.

In [ ]:
# Cell 1 — install + start SSH over Cloudflare
SSH_PASSWORD = "change-me-strong-password"  # <-- set this

!pip install -qU colab-ssh

from colab_ssh import launch_ssh_cloudflared

launch_ssh_cloudflared(
    password=SSH_PASSWORD,
    verbose=True,
)

print("\n=== Copy the ssh command above into Mac Terminal ===")

In [ ]:
# Cell 2 — keep Colab alive (run after SSH is up)
import IPython
from google.colab import output

def _keepalive():
    display(IPython.display.Javascript(
        """
        (function() {
          console.log('Colab SSH keepalive');
          setInterval(() => console.log('keepalive'), 60000);
        })();
        """
    ))

_keepalive()
print("Keepalive active — leave this notebook tab open.")

## After SSH into Colab (from Mac)

```bash
# Option A: clone
git clone https://github.com/Nontakorn-dev/DernDee_Research.git
cd DernDee_Research && pip install -r shared/requirements.txt

# Option B: scp from Mac (run on Mac, not Colab)
# scp -P PORT -r ~/Documents/Nontakorn/Research root@HOST:~/Research

# Dataset (Drive symlink example)
# ln -sf /content/drive/MyDrive/Dataset/processed/Xy dataset/Xy

nvidia-smi
bash experiments/scripts/run_kfold_colab.sh --model tcn --device cuda
```